# Rveda Training Smoke Launcher

This notebook is a thin launcher for the Task 3.3 smoke run.

It installs the runtime dependencies, checks that the repo is visible, and then calls `train_grpo_smoke.py`.
It does not duplicate the training logic.


## 1. Install runtime dependencies

Run this once per Colab session.


In [1]:
from pathlib import Path
import json
import subprocess
import sys

%pip install -q --upgrade pip
%pip install -q "openenv-core[core]>=0.2.3" datasets accelerate unsloth
print("Installed: openenv-core[core]>=0.2.3, datasets, accelerate, unsloth")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.0 MB/s eta 0:00:00
Installed: openenv-core[core]>=0.2.3, datasets, accelerate, unsloth


## 2. Confirm or clone the repo

> If the repo is not present in common Colab paths, this cell can clone it automatically.

The notebook expects `train_grpo_smoke.py` to be present in the current workspace or mounted folder.

In [2]:
import os
import subprocess
from pathlib import Path

repo_root = Path(os.environ.get("RVEDA_REPO_ROOT", Path.cwd()))
script_path = repo_root / "train_grpo_smoke.py"

if not script_path.exists():
    for candidate in (Path("/content/rveda"), Path("/content/drive/MyDrive/rveda"), Path("/workspace/rveda")):
        if (candidate / "train_grpo_smoke.py").exists():
            repo_root = candidate
            script_path = candidate / "train_grpo_smoke.py"
            break

if not script_path.exists():
    clone_target = Path("/content/rveda")
    clone_target.parent.mkdir(parents=True, exist_ok=True)
    clone_url = os.environ.get("RVEDA_GIT_URL", "https://github.com/anirudw/rveda.git")
    print(f"Repo not found locally. Cloning from: {clone_url}")
    try:
        subprocess.check_call(["git", "clone", clone_url, str(clone_target)])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Git clone failed. Set RVEDA_GIT_URL to your repo URL and rerun this cell."
        ) from exc
    repo_root = clone_target
    script_path = repo_root / "train_grpo_smoke.py"

if not script_path.exists():
    raise FileNotFoundError(
        "Could not find train_grpo_smoke.py after clone/lookup. "
        "Set RVEDA_REPO_ROOT to your repo path and rerun this cell."
    )

def run_git(args, check=True):
    proc = subprocess.run(
        ["git", "-C", str(repo_root), *args],
        text=True,
        capture_output=True,
    )
    if check and proc.returncode != 0:
        detail = (proc.stderr or "").strip() or (proc.stdout or "").strip() or f"exit code {proc.returncode}"
        raise RuntimeError(f"git {' '.join(args)} failed: {detail}")
    return proc

git_dir = repo_root / ".git"
if git_dir.exists():
    branch_override = os.environ.get("RVEDA_GIT_BRANCH", "").strip()
    try:
        remotes = [r.strip() for r in run_git(["remote"], check=False).stdout.splitlines() if r.strip()]
        if not remotes:
            print("No git remotes configured; using local checkout without fetch/pull.")
        else:
            remote = remotes[0]
            run_git(["fetch", "--all", "--prune"])
            current_branch = run_git(["rev-parse", "--abbrev-ref", "HEAD"]).stdout.strip()
            if branch_override:
                run_git(["checkout", branch_override])
                current_branch = branch_override
            if current_branch == "HEAD":
                print("Detached HEAD detected; skipping pull --ff-only. Set RVEDA_GIT_BRANCH to pull a branch.")
            else:
                run_git(["pull", "--ff-only", remote, current_branch])
                print(f"Git sync ok: {remote}/{current_branch}")
    except Exception as exc:
        print("Warning: git sync failed, continuing with local checkout.")
        print(exc)

os.environ["RVEDA_REPO_ROOT"] = str(repo_root.resolve())
%cd {repo_root}
print("Repo root:", repo_root)
print("Script path:", script_path)
print("RVEDA_REPO_ROOT:", os.environ["RVEDA_REPO_ROOT"])
if (repo_root / ".git").exists():
    rev = subprocess.check_output(["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True).strip()
    print("Repo commit:", rev)


Repo not found locally. Cloning from: https://github.com/anirudw/rveda.git
Git sync ok: origin/main
/content/rveda
Repo root: /content/rveda
Script path: /content/rveda/train_grpo_smoke.py
RVEDA_REPO_ROOT: /content/rveda
Repo commit: 424579553afec94d784dbb2fa7e8b05eef1c44cf


## 3. Launch the smoke runner

This calls the existing script with a minimal Colab-friendly configuration.


In [3]:
output_dir = repo_root / "artifacts" / "grpo_smoke_colab"
command = [
    sys.executable,
    str(script_path),
    "--model-name",
    "Qwen/Qwen2.5-7B-Instruct",
    "--output-dir",
    str(output_dir),
    "--task-ids",
    "v2_easy_overweight_schema_v1",
    "--samples-per-task",
    "1",
    "--episodes",
    "1",
    "--train-steps",
    "1",
    "--max-episode-steps",
    "2",
]
print("Running:", " ".join(command))
result = subprocess.run(command, text=True, capture_output=True)
print("\n--- stdout ---")
print(result.stdout or "<empty>")
print("\n--- stderr ---")
print(result.stderr or "<empty>")
if result.returncode != 0:
    raise RuntimeError(f"train_grpo_smoke.py failed with exit code {result.returncode}")

Running: /usr/bin/python3 /content/rveda/train_grpo_smoke.py --model-name Qwen/Qwen2.5-7B-Instruct --output-dir /content/rveda/artifacts/grpo_smoke_colab --task-ids v2_easy_overweight_schema_v1 --samples-per-task 1 --episodes 1 --train-steps 1 --max-episode-steps 2

--- stdout ---
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Un

## 4. Inspect the generated artifacts


In [4]:
summary_path = output_dir / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError(f"Missing summary artifact: {summary_path}")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))
print("Artifacts:")
for file_name in ["scripted_baseline.json", "baseline_model_eval.json", "post_train_model_eval.json", "summary.json"]:
    print("-", output_dir / file_name)

{
  "model_name": "Qwen/Qwen2.5-7B-Instruct",
  "task_ids": [
    "v2_easy_overweight_schema_v1"
  ],
  "train_rows": 1,
  "train_steps": 1,
  "baseline_mean_total_reward": 0.1,
  "post_train_mean_total_reward": 0.1,
  "baseline_rollout_summary": {
    "episode_count": 1,
    "nonzero_reward_rate": 1.0,
    "mean_total_reward": 0.1
  },
  "post_train_rollout_summary": {
    "episode_count": 1,
    "nonzero_reward_rate": 1.0,
    "mean_total_reward": 0.1
  },
  "trainer_metrics": {
    "train_runtime": 54.1042,
    "train_samples_per_second": 0.037,
    "train_steps_per_second": 0.018,
    "total_flos": 0.0,
    "train_loss": -6.208817639706543e-13
  },
  "saved_model_dir": "/content/rveda/artifacts/grpo_smoke_colab/model"
}
Artifacts:
- /content/rveda/artifacts/grpo_smoke_colab/scripted_baseline.json
- /content/rveda/artifacts/grpo_smoke_colab/baseline_model_eval.json
- /content/rveda/artifacts/grpo_smoke_colab/post_train_model_eval.json
- /content/rveda/artifacts/grpo_smoke_colab/summ

## 5. Phase 1: Archive Baseline Run

Freeze this successful smoke run into a permanent baseline folder and capture reproducibility metadata.

In [5]:
from datetime import date
import shutil

run_date = date.today().isoformat().replace("-", "_")
baseline_dir = repo_root / "artifacts" / f"run_smoke_min_{run_date}"
baseline_dir.mkdir(parents=True, exist_ok=True)

required_files = [
    "scripted_baseline.json",
    "baseline_model_eval.json",
    "post_train_model_eval.json",
    "summary.json",
]

missing = [name for name in required_files if not (output_dir / name).exists()]
if missing:
    raise FileNotFoundError(
        "Cannot complete Phase 1 because smoke artifacts are missing: " + ", ".join(missing)
    )

for name in required_files:
    src = output_dir / name
    dst = baseline_dir / name
    shutil.copy2(src, dst)

if (output_dir / "model").exists():
    model_dst = baseline_dir / "model"
    if model_dst.exists():
        shutil.rmtree(model_dst)
    shutil.copytree(output_dir / "model", model_dst)

launch_command = " ".join(command)
(baseline_dir / "launch_command.txt").write_text(launch_command + "\n", encoding="utf-8")

commit_hash = "unknown"
if (repo_root / ".git").exists():
    commit_hash = subprocess.check_output(
        ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
    ).strip()

run_metadata = {
    "phase": "phase_1_baseline_freeze",
    "date": run_date,
    "repo_root": str(repo_root),
    "source_output_dir": str(output_dir),
    "baseline_dir": str(baseline_dir),
    "commit_hash": commit_hash,
    "model_name": "Qwen/Qwen2.5-7B-Instruct",
    "task_ids": ["v2_easy_overweight_schema_v1"],
    "launcher_command": launch_command,
    "stack": {
        "unsloth": "2026.4.8",
        "transformers": "5.5.0",
        "torch": "2.10.0+cu128",
        "cuda_toolkit": "12.8",
        "gpu": "Tesla T4",
    },
}
(baseline_dir / "run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2), encoding="utf-8"
 )

print("Baseline run archived at:", baseline_dir)
print("Captured commit:", commit_hash)

Baseline run archived at: /content/rveda/artifacts/run_smoke_min_2026_04_26
Captured commit: 424579553afec94d784dbb2fa7e8b05eef1c44cf


In [6]:
required_summary_keys = [
    "model_name",
    "task_ids",
    "train_rows",
    "train_steps",
    "baseline_mean_total_reward",
    "post_train_mean_total_reward",
    "baseline_rollout_summary",
    "post_train_rollout_summary",
    "trainer_metrics",
    "saved_model_dir",
]

summary = json.loads((baseline_dir / "summary.json").read_text(encoding="utf-8"))
missing_keys = [key for key in required_summary_keys if key not in summary]
if missing_keys:
    raise KeyError("Missing summary keys: " + ", ".join(missing_keys))

phase1_status = [
    "# Phase 1 Status",
    "",
    "Status: PASS",
    f"Date: {run_date}",
    f"Commit: {commit_hash}",
    f"Baseline Folder: {baseline_dir}",
    "",
    "Checks:",
    "- Baseline artifacts copied",
    "- Summary schema validated",
    "- Launch command captured",
    "- Run metadata captured",
]
(baseline_dir / "phase1_status.md").write_text("\n".join(phase1_status) + "\n", encoding="utf-8")

print("Phase 1 complete: PASS")
print("Validated summary keys:", len(required_summary_keys))
print("Wrote:", baseline_dir / "phase1_status.md")

Phase 1 complete: PASS
Validated summary keys: 10
Wrote: /content/rveda/artifacts/run_smoke_min_2026_04_26/phase1_status.md
